# 02. 수요기업 검색 문장 정리

계속사업자만 남기고 사업자 식별자 기준 중복을 정리한 뒤, 기업 설명을 임베딩 입력 문장으로 만듭니다. 이 노트북은 비공개 원본을 읽어 비공개 산출물만 생성합니다.

In [ ]:
from pathlib import Path

import pandas as pd

INPUT_PATH = Path('../data/raw/company_master.csv')
OUTPUT_PATH = Path('../artifacts/demand_company_embedding_input.csv')
REQUIRED_COLUMNS = {'company_id', 'business_status', 'company_description'}


def build_company_embedding_text(company_df: pd.DataFrame) -> pd.DataFrame:
    missing_columns = REQUIRED_COLUMNS - set(company_df.columns)
    if missing_columns:
        raise KeyError(f'Missing required columns: {sorted(missing_columns)}')

    active_df = company_df.loc[company_df['business_status'].eq('계속사업자')].copy()
    active_df = active_df.drop_duplicates(subset='company_id', keep='last')
    active_df['embedding_text'] = (
        active_df['company_description']
        .fillna('')
        .astype(str)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )
    return active_df.loc[active_df['embedding_text'].ne('')]


if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Private input is not available: {INPUT_PATH}')

company_df = pd.read_csv(INPUT_PATH)
demand_company_df = build_company_embedding_text(company_df)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
demand_company_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
demand_company_df[['company_id', 'embedding_text']].head()